# Debug Seller Property Listing Model

This notebook is for debugging the 500 Internal Server Error returned by `GET /api/sellers/:id/properties`. The goal is to isolate the SQL or connection issue in `src/models/seller.property.listing.model.js`.

In [1]:
# 1. Import Required Libraries
from pathlib import Path
import os
import subprocess
import textwrap

workspace_root = Path(r"c:\Users\hp\Desktop\real-kudu-api2")
print('Workspace root:', workspace_root)
print('Python version:', os.sys.version)

Workspace root: c:\Users\hp\Desktop\real-kudu-api2
Python version: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]


In [2]:
# 2. Set Up Database Connection
dotenv_path = workspace_root / '.env'
print('Env file exists:', dotenv_path.exists())

if dotenv_path.exists():
    from dotenv import load_dotenv
    load_dotenv(dotenv_path)

print('DATABASE_URL:', os.getenv('DATABASE_URL'))

Env file exists: True


ModuleNotFoundError: No module named 'dotenv'

In [ ]:
# 3. Define Property Types and Sort Fields
PROPERTY_TYPES = {
    'all': 'all',
    'estate': 'estate',
    'house': 'house',
    'house_for_sale': 'house_for_sale',
    'land': 'land',
}

SORT_FIELDS = {
    'created_at': 'created_at',
    'price': 'price',
    'name': 'name',
    'property_type': 'property_type',
}

print('Property types:', PROPERTY_TYPES)
print('Sort fields:', SORT_FIELDS)

In [ ]:
# 4. Build Property Union Query
def build_property_union(seller_id, property_type='all'):
    include_estate = property_type == PROPERTY_TYPES['all'] or property_type == PROPERTY_TYPES['estate']
    include_house = property_type == PROPERTY_TYPES['all'] or property_type == PROPERTY_TYPES['house']
    include_house_for_sale = property_type == PROPERTY_TYPES['all'] or property_type == PROPERTY_TYPES['house_for_sale']
    include_land = property_type == PROPERTY_TYPES['all'] or property_type == PROPERTY_TYPES['land']

    values = [seller_id]
    subqueries = []

    if include_estate:
        subqueries.append(textwrap.dedent('''
            SELECT
                'estate'::text AS property_type,
                e.id AS id,
                COALESCE(e.name, '') AS name,
                COALESCE(e.address, '') AS location,
                NULL::numeric AS price,
                e.seller_id AS seller_id,
                e.created_at,
                e.updated_at,
                jsonb_build_object(
                    'estateType', e.estate_type,
                    'isLandEstate', e.is_land_estate,
                    'lga', e.lga,
                    'state', e.state,
                    'coverImageUrl', e.cover_image_url
                ) AS details
            FROM estates e
            WHERE e.seller_id = $1 AND e.deleted_at IS NULL
        '''))

    if include_house:
        subqueries.append(textwrap.dedent('''
            SELECT
                'house'::text AS property_type,
                h.id AS id,
                COALESCE(h.name, CONCAT('House - ', COALESCE(h.address, 'Unknown'))) AS name,
                COALESCE(h.address, '') AS location,
                NULL::numeric AS price,
                h.seller_id AS seller_id,
                h.created_at,
                h.updated_at,
                jsonb_build_object(
                    'estateId', h.estate_id,
                    'isSingleHouse', h.is_single_house,
                    'type', h.type,
                    'state', h.state,
                    'lga', h.lga,
                    'description', h.house_description,
                    'images', h.images
                ) AS details
            FROM houses h
            WHERE h.seller_id = $1 AND h.deleted_at IS NULL
        '''))

    if include_house_for_sale:
        subqueries.append(textwrap.dedent('''
            SELECT
                'house_for_sale'::text AS property_type,
                h.house_id AS id,
                COALESCE(h.address, '') AS name,
                COALESCE(h.address, '') AS location,
                h.asking_price::numeric AS price,
                h.owner_id AS seller_id,
                h.created_at,
                h.updated_at,
                jsonb_build_object(
                    'houseType', h.house_type,
                    'status', h.status,
                    'bedrooms', h.bedrooms,
                    'state', h.state,
                    'lga', h.lga,
                    'description', h.description,
                    'images', h.images,
                    'verificationStatus', h.verification_status
                ) AS details
            FROM houses_for_sale h
            WHERE h.owner_id = $1
        '''))

    if include_land:
        subqueries.append(textwrap.dedent('''
            SELECT
                'land'::text AS property_type,
                l.property_id AS id,
                COALESCE(l.property_name, CONCAT('Land - ', COALESCE(l.property_address, l.state_location, 'Unknown'))) AS name,
                COALESCE(l.property_address, l.state_location, '') AS location,
                l.price::numeric AS price,
                l.seller_id AS seller_id,
                l.created_at,
                l.updated_at,
                jsonb_build_object(
                    'estateId', l.estate_id,
                    'isEstateLand', l.is_estate_land,
                    'landType', l.land_type,
                    'state', l.state_location,
                    'description', COALESCE(l.long_description, l.short_description),
                    'images', l.gallery_images
                ) AS details
            FROM land_properties l
            WHERE l.seller_id = $1
        '''))

    return '
UNION ALL
'.join(subqueries), values

query, vals = build_property_union('8db54fcf-a7dc-489c-82a1-845c22271f1b')
print('Query length:', len(query))
print(query[:500])
print('Values:', vals)

In [ ]:
# 5. Parse Sort Parameters
def parse_sort(sort_by='created_at', sort_order='desc'):
    column = SORT_FIELDS.get(str(sort_by).lower(), SORT_FIELDS['created_at'])
    order = 'ASC' if str(sort_order).lower() == 'asc' else 'DESC'
    return column, order

print(parse_sort('created_at', 'desc'))
print(parse_sort('price', 'asc'))
print(parse_sort('property_type', 'desc'))

In [ ]:
# 6. Execute Find All By Seller Query
node_script = r"""
import 'dotenv/config';
import SellerPropertyListingModel from './src/models/seller.property.listing.model.js';

(async () => {
    try {
        const result = await SellerPropertyListingModel.findAllBySeller('8db54fcf-a7dc-489c-82a1-845c22271f1b', { page: 1, limit: 10 });
        console.log(JSON.stringify({ success: true, result }, null, 2));
    } catch (error) {
        console.error(JSON.stringify({ success: false, error: {
            message: error.message,
            stack: error.stack,
            code: error.code
        } }, null, 2));
        process.exit(1);
    }
})();
"""
script_path = workspace_root / 'src' / 'tmp_node_seller_debug.mjs'
script_path.write_text(node_script, encoding='utf-8')
print('Written temporary Node script to', script_path)

proc = subprocess.run(['node', str(script_path)], cwd=workspace_root, capture_output=True, text=True)
print('RETURN CODE:', proc.returncode)
print('STDOUT:
', proc.stdout)
print('STDERR:
', proc.stderr)

In [ ]:
# 7. Normalize Query Results
def normalize_row(row):
    return {
        'id': row['id'],
        'propertyType': row['property_type'],
        'name': row['name'],
        'location': row['location'],
        'price': None if row['price'] is None else float(row['price']),
        'sellerId': row['seller_id'],
        'createdAt': row['created_at'],
        'updatedAt': row['updated_at'],
        'details': row['details'],
    }

example_row = {
    'id': '000',
    'property_type': 'estate',
    'name': 'Test',
    'location': 'Nowhere',
    'price': None,
    'seller_id': '8db54fcf-a7dc-489c-82a1-845c22271f1b',
    'created_at': '2026-01-01',
    'updated_at': '2026-01-02',
    'details': {'foo': 'bar'},
}
print(normalize_row(example_row))

In [ ]:
# 8. Handle Query Errors
print('If the query still fails, inspect the SQL syntax, table names, and parameter placeholders.')
print('Common problems: invalid column names, missing fields in DB schema, or driver connection issues.')